In [98]:
import torch
import math
from dataclasses import dataclass

def nearest_multiple_of_64(d_model: int) -> int:
    value = 8/3 * d_model 
    return round(value / 64) * 64
 
def flops(m: int, n: int, p: int) -> int:
    return 2*m*n*p
    
def flops_matmul(A: tuple[int, ...], B: tuple[int, ...]) -> int:
    batch_dims = A[:-2]

    b = math.prod(batch_dims)
    
    m, n = A[-2:]
    n2, p = B[-2:]
    assert n == n2, "Invalid matmul dims"
    return b * flops(m, n, p)

@dataclass
class ModelConfig:
    batch_size: int
    vocab_size: int 
    context_length: int
    num_layers: int
    d_model: int
    num_heads: int
    d_ff: int


    @property
    def parameter_bytes(self):
        return 4
    

    @property
    def d_k(self):
        return self.d_model // self.num_heads

    def parameters(self):
        # Total trainable params
        embeddings = self.vocab_size * self.d_model
        
        # Transformer Block
        rms_norm1 = self.d_model 
        attention = (self.d_model ** 2) * 4 # for Q, K, V, O
        rms_norm2 = self.d_model 
        swiglu_ff = self.d_model * self.d_ff * 3
        transformer_block = self.num_layers * (rms_norm1 + attention + rms_norm2 + swiglu_ff)
        
        # head
        layer_norm = self.d_model
        lm_head = self.d_model * self.vocab_size
        
        # language model total
        trainable_params = embeddings + transformer_block + layer_norm + lm_head
        return trainable_params

    def memory(self):
        return self.parameters() * self.parameter_bytes

    def flops(self):
        batch_size = self.batch_size
        context_length = self.context_length
        d_model = self.d_model
        num_layers = self.num_layers
        d_k = self.d_k
        d_ff = self.d_ff
        
        # Q = x @ Wq
        x_shape = (batch_size, context_length, d_model)
        Wq_shape = (d_model, d_model)
        flops_q_proj = flops_matmul(x_shape, Wq_shape) * num_layers
        print(f"flops_q_proj = {flops_q_proj:,}")
        
        # K = x @ Wk
        x_shape = (batch_size, context_length, d_model)
        Wk_shape = (d_model, d_model)
        flops_k_proj = flops_matmul(x_shape, Wk_shape) * num_layers
        print(f"flops_k_proj = {flops_k_proj:,}")
        
        # V = x @ Wv
        x_shape = (batch_size, context_length, d_model)
        Wv_shape = (d_model, d_model)
        flops_v_proj = flops_matmul(x_shape, Wv_shape) * num_layers
        print(f"flops_v_proj = {flops_v_proj:,}")
        
        # Scaled dot-product attention
        # softmax(Q @ K_T / sqrt(d_k)) @ V
        Q_shape = (batch_size, num_heads, context_length, d_k) 
        KT_shape = (batch_size, num_heads, d_k, context_length) 
        V_shape = (batch_size, num_heads, context_length, d_k)
        
        flops_q_kt_proj = flops_matmul(Q_shape, KT_shape) * num_layers
        print(f"flops_q_kt_proj = {flops_q_kt_proj:,}")
        
        # out = attn @ V
        attn_shape = (batch_size, num_heads, context_length, context_length)
        V_shape = (batch_size, num_heads, context_length, d_k)
        flops_attn_v_proj = flops_matmul(attn_shape, V_shape) * num_layers
        print(f"flops_attn_v_proj = {flops_attn_v_proj:,}")
        
        # out_proj = attn @ Wo
        attn_out_shape = (batch_size, context_length, d_model)
        Wo_shape = (d_model, d_model)
        flops_out_proj = flops_matmul(attn_out_shape, Wo_shape) * num_layers
        print(f"flops_out_proj = {flops_out_proj:,}")
        
        # SwiGLU FF
        x_shape = (batch_size, context_length, d_model)
        w1_shape = (d_model, d_ff)
        w1_proj_flops = flops_matmul(x_shape, w1_shape) * num_layers
        print(f"w1_proj_flops = {w1_proj_flops:,}")
        w3_shape = (d_model, d_ff)
        w3_proj_flops = flops_matmul(x_shape, w3_shape) * num_layers
        print(f"w3_proj_flops = {w3_proj_flops:,}")
        x_shape = (batch_size, context_length, d_ff)
        w2_shape = (d_ff, d_model)
        w2_proj_flops = flops_matmul(x_shape, w2_shape) * num_layers
        print(f"w2_proj_flops = {w2_proj_flops:,}")
        
        # LM head
        x_shape = (batch_size, context_length, d_model)
        lm_head_shape = (d_model, vocab_size)
        flops_lm_head = flops_matmul(x_shape, lm_head_shape)
        print(f"flops_lm_head = {flops_lm_head:,}")
        
        total_flops = flops_q_proj + flops_k_proj + flops_v_proj + flops_q_kt_proj + flops_attn_v_proj + flops_out_proj + w1_proj_flops + w3_proj_flops + w2_proj_flops + flops_lm_head

        return total_flops

    def summary(self, name: str):
        trainable_params = self.parameters()
        memory_bytes = self.memory() 
        memory_gib = memory_bytes / 1024**3
        
        print(f"{name} trainable parameters: {trainable_params:,}")
        print(f"total_memory: {memory_gib:.2f} GiB")
        print(f"total_flops: {self.flops():.1e}")
        
    
        

gpt2_xl = ModelConfig(
    batch_size=1, 
    vocab_size = 50_257,
    context_length = 1024,
    num_layers = 48,
    d_model = 1600,
    num_heads = 25,
    d_ff = 4288
) 

gpt2_xl.summary("GPT-2 XL") 

GPT-2 XL trainable parameters: 1,640,452,800
total_memory: 6.11 GiB
flops_q_proj = 251,658,240,000
flops_k_proj = 251,658,240,000
flops_v_proj = 251,658,240,000
flops_q_kt_proj = 161,061,273,600
flops_attn_v_proj = 161,061,273,600
flops_out_proj = 251,658,240,000
w1_proj_flops = 674,444,083,200
w3_proj_flops = 674,444,083,200
w2_proj_flops = 674,444,083,200
flops_lm_head = 164,682,137,600
total_flops: 3.5e+12


In [99]:
    
gpt2_small = ModelConfig(
    batch_size=1, 
    vocab_size = 50_257,
    context_length = 1024,
    num_layers = 12,
    d_model = 768,
    num_heads = 12,
    d_ff = nearest_multiple_of_64(768) 
)
print(gpt2_small)
gpt2_small.summary("GPT2-Small")


ModelConfig(batch_size=1, vocab_size=50257, context_length=1024, num_layers=12, d_model=768, num_heads=12, d_ff=2048)
GPT2-Small trainable parameters: 162,148,608
total_memory: 0.60 GiB
flops_q_proj = 14,495,514,624
flops_k_proj = 14,495,514,624
flops_v_proj = 14,495,514,624
flops_q_kt_proj = 40,265,318,400
flops_attn_v_proj = 40,265,318,400
flops_out_proj = 14,495,514,624
w1_proj_flops = 38,654,705,664
w3_proj_flops = 38,654,705,664
w2_proj_flops = 38,654,705,664
flops_lm_head = 79,047,426,048
total_flops: 3.3e+11


In [100]:
gpt2_med = ModelConfig(
    batch_size=1, 
    vocab_size = 50_257,
    context_length = 1024,
    num_layers = 24,
    d_model = 1024,
    num_heads = 16,
    d_ff = nearest_multiple_of_64(1024) 
)
print(gpt2_med)
gpt2_med.summary("GPT2-Medium")


ModelConfig(batch_size=1, vocab_size=50257, context_length=1024, num_layers=24, d_model=1024, num_heads=16, d_ff=2752)
GPT2-Medium trainable parameters: 406,539,264
total_memory: 1.51 GiB
flops_q_proj = 51,539,607,552
flops_k_proj = 51,539,607,552
flops_v_proj = 51,539,607,552
flops_q_kt_proj = 80,530,636,800
flops_attn_v_proj = 80,530,636,800
flops_out_proj = 51,539,607,552
w1_proj_flops = 138,512,695,296
w3_proj_flops = 138,512,695,296
w2_proj_flops = 138,512,695,296
flops_lm_head = 105,396,568,064
total_flops: 8.9e+11


In [102]:
gpt2_large = ModelConfig(
    batch_size=1, 
    vocab_size = 50_257,
    context_length = 1024,
    num_layers = 36,
    d_model = 1280,
    num_heads = 20,
    d_ff = nearest_multiple_of_64(1280) 
)
print(gpt2_large)
gpt2_large.summary("GPT2-Large")


ModelConfig(batch_size=1, vocab_size=50257, context_length=1024, num_layers=36, d_model=1280, num_heads=20, d_ff=3392)
GPT2-Large trainable parameters: 833,591,040
total_memory: 3.11 GiB
flops_q_proj = 120,795,955,200
flops_k_proj = 120,795,955,200
flops_v_proj = 120,795,955,200
flops_q_kt_proj = 120,795,955,200
flops_attn_v_proj = 120,795,955,200
flops_out_proj = 120,795,955,200
w1_proj_flops = 320,109,281,280
w3_proj_flops = 320,109,281,280
w2_proj_flops = 320,109,281,280
flops_lm_head = 131,745,710,080
total_flops: 1.8e+12


In [103]:
gpt2_xl_large_ctx = ModelConfig(
    batch_size=1, 
    vocab_size = 50_257,
    context_length = 16_384,
    num_layers = 48,
    d_model = 1600,
    num_heads = 25,
    d_ff = nearest_multiple_of_64(1600) 
)
print(gpt2_xl_large_ctx)
gpt2_xl_large_ctx.summary("GPT2-XL 16,384 context")


ModelConfig(batch_size=1, vocab_size=50257, context_length=16384, num_layers=48, d_model=1600, num_heads=25, d_ff=4288)
GPT2-XL 16,384 context trainable parameters: 1,640,452,800
total_memory: 6.11 GiB
flops_q_proj = 4,026,531,840,000
flops_k_proj = 4,026,531,840,000
flops_v_proj = 4,026,531,840,000
flops_q_kt_proj = 41,231,686,041,600
flops_attn_v_proj = 41,231,686,041,600
flops_out_proj = 4,026,531,840,000
w1_proj_flops = 10,791,105,331,200
w3_proj_flops = 10,791,105,331,200
w2_proj_flops = 10,791,105,331,200
flops_lm_head = 2,634,914,201,600
total_flops: 1.3e+14
